# Recurrent Neural Network(RNN)

### Sunny Rainy example

In [1]:
import torch
import torch.nn as nn

# Linear model: 2 inputs → 2 outputs
model = nn.Linear(in_features=2, out_features=2, bias=False)

# Initialize weight matrix manually
with torch.no_grad():
    model.weight.copy_(torch.tensor([
        [1.0, 0],
        [0,  1.0]
    ]))

# Example input (batch of 1 sample)
x1 = torch.tensor([[0.0, 1]]) # rainy
x2 = torch.tensor([[1.0, 0.0]]) # sunny

for x in [x1,x2]:
    y = model(x) 
    print(y)

tensor([[0., 1.]], grad_fn=<MmBackward0>)
tensor([[1., 0.]], grad_fn=<MmBackward0>)


### Detailed example

In [2]:
import random
import torch
import torch.nn as nn
import torch.optim as optim

### 1) Generate dataset

In [3]:

dishes = ['A', 'B', 'C']
weather_types = ['Sunny', 'Rainy']

dish_to_idx = {d: i for i, d in enumerate(dishes)}
weather_to_idx = {w: i for i, w in enumerate(weather_types)}

def next_dish(dish):
    if dish == 'A':
        return 'B'
    elif dish == 'B':
        return 'C'
    else:
        return 'A'

def generate_sequence(length=1000):
    """
    Returns:
        inputs  = [(dish_t, weather_t), ...]
        targets = [dish_{t+1}, ...]
    """
    current_dish = 'A'

    inputs = []
    targets = []

    for _ in range(length):

        weather = random.choice(weather_types)

        inputs.append((current_dish, weather))

        if weather == 'Sunny':
            new_dish = current_dish
        else:  # Rainy
            new_dish = next_dish(current_dish)

        targets.append(new_dish)

        current_dish = new_dish

    return inputs, targets

### 2) Encode data

In [4]:
def encode_input(dish, weather):
    """
    One-hot encode dish (3 dims)
    +
    One-hot encode weather (2 dims)

    Total input size = 5
    """
    x = torch.zeros(5)

    x[dish_to_idx[dish]] = 1.0
    x[3 + weather_to_idx[weather]] = 1.0

    return x


inputs, targets = generate_sequence(length=2000)

X = torch.stack([
    encode_input(d, w)
    for d, w in inputs
])

y = torch.tensor([
    dish_to_idx[t]
    for t in targets
])

# RNN expects:
# (batch_size, seq_len, input_size)

X = X.unsqueeze(0)      # (1, seq_len, 5)
y = y.unsqueeze(0)      # (1, seq_len)

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: torch.Size([1, 2000, 5])
y shape: torch.Size([1, 2000])


### 3) Vanilla RNN Model

In [5]:
class DishRNN(nn.Module):
    def __init__(self,
                 input_size=5,
                 hidden_size=3,
                 output_size=3):
        super().__init__()

        self.rnn = nn.RNN(
            input_size=input_size,
            hidden_size=hidden_size,
            batch_first=True, 
            bias=False, 
            nonlinearity = "tanh"
        )

        self.fc = nn.Linear(hidden_size, output_size, bias=False)

    def forward(self, x):
        # rnn_out:
        # (batch, seq_len, hidden_size)

        rnn_out, hidden = self.rnn(x)

        logits = self.fc(rnn_out)

        return logits


model = DishRNN()

In [6]:
model

DishRNN(
  (rnn): RNN(5, 3, bias=False, batch_first=True)
  (fc): Linear(in_features=3, out_features=3, bias=False)
)

### 4) Training Setup

In [7]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

### 5) Traning loop

In [8]:
epochs = 300

for epoch in range(epochs):

    optimizer.zero_grad()

    logits = model(X)

    # reshape for CE loss
    loss = criterion(
        logits.view(-1, 3),
        y.view(-1)
    )

    loss.backward()
    optimizer.step()

    if (epoch + 1) % 50 == 0:
        print(
            f"Epoch [{epoch+1}/{epochs}] "
            f"Loss = {loss.item():.6f}"
        )

Epoch [50/300] Loss = 0.830262
Epoch [100/300] Loss = 0.406373
Epoch [150/300] Loss = 0.127197
Epoch [200/300] Loss = 0.063532
Epoch [250/300] Loss = 0.040274
Epoch [300/300] Loss = 0.028575


### 6) Evaluation

In [9]:
with torch.no_grad():

    logits = model(X)

    preds = logits.argmax(dim=-1)

    accuracy = (
        (preds == y).float().mean().item()
    )

print("\nAccuracy:", accuracy)


Accuracy: 1.0


### 7) Test on all possible transitions

In [10]:
print("\nLearned transitions:\n")

test_cases = [
    ('A', 'Sunny'),
    ('A', 'Rainy'),
    ('B', 'Sunny'),
    ('B', 'Rainy'),
    ('C', 'Sunny'),
    ('C', 'Rainy'),
]

hidden = None

for dish, weather in test_cases:

    x = encode_input(dish, weather)
    x = x.unsqueeze(0).unsqueeze(0)

    with torch.no_grad():
        logits = model(x)
        pred = logits.argmax(-1).item()

    print(
        f"Current Dish={dish:1s}, "
        f"Weather={weather:5s} "
        f"--> Predicted Next Dish={dishes[pred]}"
    )


Learned transitions:

Current Dish=A, Weather=Sunny --> Predicted Next Dish=A
Current Dish=A, Weather=Rainy --> Predicted Next Dish=B
Current Dish=B, Weather=Sunny --> Predicted Next Dish=B
Current Dish=B, Weather=Rainy --> Predicted Next Dish=C
Current Dish=C, Weather=Sunny --> Predicted Next Dish=C
Current Dish=C, Weather=Rainy --> Predicted Next Dish=A


In [11]:
for name, param in model.named_parameters():
    print(name)
    print(param.data)
    print()

rnn.weight_ih_l0
tensor([[ 1.0395, -0.1569, -1.7484,  1.6756, -1.3781],
        [-0.5044, -2.1152,  1.9922,  1.3201, -1.4874],
        [-1.2882,  2.2354, -0.0713,  1.4919, -1.0725]])

rnn.weight_hh_l0
tensor([[ 0.7325,  0.1354, -0.6457],
        [-0.8727,  1.2143,  1.4129],
        [ 0.1843, -0.1278, -0.2706]])

fc.weight
tensor([[ 1.0389,  2.8380, -2.6130],
        [ 1.9766, -2.7816, -1.4812],
        [-2.5830,  0.3021,  2.3184]])



### Discussion
The RNN model achieved high accuracy with a low loss value, indicating that it learned the sequence patterns effectively. The hidden_size parameter determines the number of neurons in the hidden layer, which is responsible for retaining information from previous inputs. In this experiment, a hidden_size of 3 provided sufficient capacity for the model to capture the underlying sequence patterns and generate accurate predictio

### Conclusion

In conclusion, the minimal Vanilla RNN with a hidden size of 3 was sufficient to model the discrete finite-state machine. By using matrix multiplication and the $\tanh$ activation function, the network successfully learned the sequential dependencies and accurately captured the conditional logic of the input sequence. The results demonstrate that even a simple recurrent neural network can effectively self-organize its parameters to solve rule-based sequential prediction tasks.